# Datamart Processing

Purpose: build the bronze, silver, and gold datamart layers with Spark DataFrames.

In [13]:
from pathlib import Path
import json
import os
import sys

import pandas as pd
import matplotlib.pyplot as plt
import pyspark
import seaborn as sns
from pyspark.sql import SparkSession, functions as F, Window

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists() and (PROJECT_ROOT.parent / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data"
DATAMART_DIR = PROJECT_ROOT / "datamart"
MODEL_BANK_DIR = PROJECT_ROOT / "outputs" / "model_bank"
REPORT_DIR = PROJECT_ROOT / "outputs" / "reports"

def spark_path(path):
    return Path(path).resolve().as_posix()

spark = (pyspark.sql.SparkSession.builder
         .appName("assignment_2_notebook")
         .master("local[*]")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

print(f"Project root: {PROJECT_ROOT}")
print(f"Spark version: {spark.version}")

Project root: /opt/airflow/project
Spark version: 3.5.1


In [14]:
RUN_SPARK_DATAMART = True
DPD_CUTOFF = 30
MOB_CUTOFF = 6

def write_parquet_dir(df, path):
    df.coalesce(1).write.mode("overwrite").parquet(spark_path(path))
    print(f"Saved {path}: {df.count():,} rows")

def read_csv(path):
    return spark.read.option("header", True).option("inferSchema", True).csv(spark_path(path))

def nullify_common(df):
    return df.replace(["_", "NA", "na", "N/A", "_______", ""], None)

def clean_numeric(column_name):
    return F.regexp_replace(F.col(column_name).cast("string"), r"[^0-9.\-]", "").cast("double")

In [15]:
sources = {"clickstream": DATA_DIR / "feature_clickstream.csv",
           "attributes": DATA_DIR / "features_attributes.csv",
           "financials": DATA_DIR / "features_financials.csv",
           "loans": DATA_DIR / "lms_loan_daily.csv"}

bronze = {name: read_csv(path) for name, path in sources.items()}
for name, df in bronze.items():
    print(f"bronze/{name}: {df.count():,} rows")
    if RUN_SPARK_DATAMART:
        write_parquet_dir(df, DATAMART_DIR / "bronze" / name)

bronze/clickstream: 215,376 rows


Saved /opt/airflow/project/datamart/bronze/clickstream: 215,376 rows
bronze/attributes: 12,500 rows
Saved /opt/airflow/project/datamart/bronze/attributes: 12,500 rows
bronze/financials: 12,500 rows


Saved /opt/airflow/project/datamart/bronze/financials: 12,500 rows
bronze/loans: 137,500 rows


Saved /opt/airflow/project/datamart/bronze/loans: 137,500 rows


In [16]:
financial_numeric_cols = ["Annual_Income", "Monthly_Inhand_Salary", "Num_Bank_Accounts", "Num_Credit_Card",
                          "Interest_Rate", "Num_of_Loan", "Delay_from_due_date", "Num_of_Delayed_Payment",
                          "Changed_Credit_Limit", "Num_Credit_Inquiries", "Outstanding_Debt",
                          "Credit_Utilization_Ratio", "Total_EMI_per_month", "Amount_invested_monthly",
                          "Monthly_Balance"]

valid_payment_behaviours = ["High_spent_Large_value_payments", "High_spent_Medium_value_payments",
                            "High_spent_Small_value_payments", "Low_spent_Large_value_payments",
                            "Low_spent_Medium_value_payments", "Low_spent_Small_value_payments"]

financials_clean = nullify_common(bronze["financials"]).withColumn("snapshot_date", F.to_date("snapshot_date"))
for col in financial_numeric_cols:
    financials_clean = financials_clean.withColumn(col, clean_numeric(col))
financials_clean = (
    financials_clean
    .withColumn("Payment_Behaviour", F.when(F.col("Payment_Behaviour").isin(valid_payment_behaviours), F.col("Payment_Behaviour")))
    .where(F.col("Customer_ID").isNotNull() & F.col("snapshot_date").isNotNull()))

attributes_clean = (nullify_common(bronze["attributes"])
                    .withColumn("snapshot_date", F.to_date("snapshot_date"))
                    .withColumn("Age", clean_numeric("Age"))
                    .withColumn("Age", F.when((F.col("Age") > 0) & (F.col("Age") < 100), F.col("Age")))
                    .withColumn("SSN", F.when(F.col("SSN").rlike(r"^\d{3}-\d{2}-\d{4}$"), F.col("SSN")))
                    .where(F.col("Customer_ID").isNotNull() & F.col("snapshot_date").isNotNull()))

clickstream_clean = nullify_common(bronze["clickstream"]).withColumn("snapshot_date", F.to_date("snapshot_date"))
for i in range(1, 21):
    clickstream_clean = clickstream_clean.withColumn(f"fe_{i}", F.col(f"fe_{i}").cast("double"))
clickstream_clean = clickstream_clean.where(F.col("Customer_ID").isNotNull() & F.col("snapshot_date").isNotNull())

loans_clean = (bronze["loans"].withColumn("loan_start_date", F.to_date("loan_start_date")).withColumn("snapshot_date", F.to_date("snapshot_date")))

for col in ["tenure", "installment_num", "loan_amt", "due_amt", "paid_amt", "overdue_amt", "balance"]:
    loans_clean = loans_clean.withColumn(col, F.col(col).cast("double"))

loans_clean = (loans_clean.withColumn("mob", F.col("installment_num").cast("int"))
               .withColumn("installments_missed", F.when(F.col("due_amt") > 0, F.ceil(F.coalesce(F.col("overdue_amt"), F.lit(0.0)) / F.col("due_amt"))).otherwise(F.lit(0)))
               .withColumn("dpd", F.when(F.coalesce(F.col("overdue_amt"), F.lit(0.0)) > 0, F.col("installments_missed") * 30).otherwise(F.lit(0)).cast("int"))
               .where(F.col("Customer_ID").isNotNull() & F.col("snapshot_date").isNotNull()))

silver_tables = {"financials_clean": financials_clean,
                 "attributes_clean": attributes_clean,
                 "clickstream_clean": clickstream_clean,
                 "loans_clean": loans_clean}

for name, df in silver_tables.items():
    print(f"silver/{name}: {df.count():,} rows")
    if RUN_SPARK_DATAMART:
        write_parquet_dir(df, DATAMART_DIR / "silver" / name)

silver/financials_clean: 12,500 rows


Saved /opt/airflow/project/datamart/silver/financials_clean: 12,500 rows
silver/attributes_clean: 12,500 rows
Saved /opt/airflow/project/datamart/silver/attributes_clean: 12,500 rows
silver/clickstream_clean: 215,376 rows


Saved /opt/airflow/project/datamart/silver/clickstream_clean: 215,376 rows
silver/loans_clean: 137,500 rows


Saved /opt/airflow/project/datamart/silver/loans_clean: 137,500 rows


In [17]:
def cap_outliers(df, columns):
    result = df
    for col in columns:
        quantiles = result.approxQuantile(col, [0.01, 0.95], 0.01)
        if len(quantiles) == 2:
            lower, upper = quantiles
            result = result.withColumn(col, F.when(F.col(col) < lower, F.lit(lower)).when(F.col(col) > upper, F.lit(upper)).otherwise(F.col(col)))
    return result

financials_features = (financials_clean.fillna({"Credit_Mix": "Unknown", "Payment_Behaviour": "Unknown", "Payment_of_Min_Amount": "Unknown"})
                       .fillna({"Changed_Credit_Limit": 0.0}))
median_debt = financials_features.approxQuantile("Outstanding_Debt", [0.5], 0.01)[0]
financials_features = financials_features.fillna({"Outstanding_Debt": median_debt})
financials_features = cap_outliers(financials_features, ["Num_Bank_Accounts", "Num_Credit_Card", "Interest_Rate", "Num_of_Delayed_Payment", "Num_Credit_Inquiries"])

financials_features = (financials_features
                       .withColumn("credit_history_years", F.regexp_extract("Credit_History_Age", r"(\d+)\s+Years?", 1).cast("double"))
                       .withColumn("credit_history_month_part", F.regexp_extract("Credit_History_Age", r"(\d+)\s+Months?", 1).cast("double"))
                       .withColumn("credit_history_months", F.coalesce(F.col("credit_history_years"), F.lit(0.0)) * 12 + F.coalesce(F.col("credit_history_month_part"), F.lit(0.0)))
                       .withColumn("loan_type_clean", F.regexp_replace(F.coalesce(F.col("Type_of_Loan"), F.lit("")), r",\s*and\s*|\s+and\s+", ","))
                       .withColumn("num_loan_types", F.when(F.length("loan_type_clean") == 0, F.lit(0)).otherwise(F.size(F.split("loan_type_clean", r",\s*"))))
                       .withColumn("log_Annual_Income", F.log1p(F.greatest(F.col("Annual_Income"), F.lit(0.0))))
                       .withColumn("debt_to_income", F.col("Outstanding_Debt") / (F.coalesce(F.col("Annual_Income"), F.lit(0.0)) + 1))
                       .withColumn("emi_to_salary", F.col("Total_EMI_per_month") / (F.coalesce(F.col("Monthly_Inhand_Salary"), F.lit(0.0)) + 1))
                       .withColumn("investment_rate", F.col("Amount_invested_monthly") / (F.coalesce(F.col("Monthly_Inhand_Salary"), F.lit(0.0)) + 1))
                       .withColumn("has_credit_limit_change", (F.col("Changed_Credit_Limit") != 0).cast("int"))
                       .withColumn("balance_to_debt", (F.coalesce(F.col("Monthly_Balance"), F.lit(0.0)) + 1) / (F.coalesce(F.col("Outstanding_Debt"), F.lit(0.0)) + 1))
                       .withColumn("inq_per_loan", F.col("Num_Credit_Inquiries") / (F.coalesce(F.col("Num_of_Loan"), F.lit(0.0)) + 1))
                       .drop("credit_history_years", "credit_history_month_part", "loan_type_clean"))

attributes_features = (attributes_clean.fillna({"Occupation": "Unknown"})
                       .withColumn("Age", F.when(F.col("Age").between(18, 75), F.col("Age")))
                       .withColumn("Age_group",
                                   F.when(F.col("Age") < 25, "under_25")
                                   .when(F.col("Age") < 40, "25_39")
                                   .when(F.col("Age") < 60, "40_59")
                                   .when(F.col("Age").isNotNull(), "60_plus")
                                   .otherwise("Unknown")))

label_store = (loans_clean.where(F.col("mob") == MOB_CUTOFF)
               .withColumn("label", (F.col("dpd") >= DPD_CUTOFF).cast("int"))
               .withColumn("label_def", F.lit(f"{DPD_CUTOFF}dpd_{MOB_CUTOFF}mob"))
               .withColumnRenamed("snapshot_date", "label_snapshot_date")
               .select("Customer_ID", "loan_id", "label", "label_def", "label_snapshot_date"))

fin_for_join = (financials_features
                .withColumnRenamed("snapshot_date", "feature_snapshot_date")
                .withColumn("label_snapshot_date", F.add_months("feature_snapshot_date", MOB_CUTOFF)))

attrs_for_join = attributes_features.withColumnRenamed("snapshot_date", "feature_snapshot_date")
click_for_join = clickstream_clean.withColumnRenamed("snapshot_date", "feature_snapshot_date")

feature_store = (fin_for_join.join(label_store, ["Customer_ID", "label_snapshot_date"], "inner")
                 .join(attrs_for_join.drop("Name", "SSN"), ["Customer_ID", "feature_snapshot_date"], "inner")
                 .join(click_for_join, ["Customer_ID", "feature_snapshot_date"], "inner")
                 .drop("Annual_Income", "Monthly_Inhand_Salary", "Outstanding_Debt", "Total_EMI_per_month", "Amount_invested_monthly", "Type_of_Loan", "Credit_History_Age"))

print(f"gold/label_store: {label_store.count():,} rows")
print(f"gold/feature_store: {feature_store.count():,} rows")

if RUN_SPARK_DATAMART:
    write_parquet_dir(label_store, DATAMART_DIR / "gold" / "label_store")
    write_parquet_dir(feature_store, DATAMART_DIR / "gold" / "feature_store")

gold/label_store: 12,500 rows
gold/feature_store: 8,974 rows


Saved /opt/airflow/project/datamart/gold/label_store: 12,500 rows


Saved /opt/airflow/project/datamart/gold/feature_store: 8,974 rows
